# Flatten Token Experiment - DeBERTa v3

## Current Approach (Pooling)
Each sentence → T tokens × H dimensions → **Pooling (mean/max/cls)** → **1 vector of H dimensions**

Example: `[T=128, H=768]` → pooling → `[H=768]`

## New Approach (Flatten)
Each sentence → T tokens × H dimensions → **Flatten** → **1 vector of T×H dimensions**

Example: `[T=128, H=768]` → flatten → `[T×H=98,304]`

**Goal**: Preserve ALL token information without aggregation, creating a high-dimensional representation per sentence.

**Model**: Using `RazyDave/deberta-v3-base-finetuned-rte` (DeBERTa-v3-base fine-tuned on RTE)

## Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from features_extraction import (
    FeaturesExtraction,
    ExtractionConfig,
    setup_logging,
)

setup_logging(level="INFO")
print("✓ Setup complete")

## Load Model and Data

In [ ]:
# Load DeBERTa-v3-base fine-tuned on RTE
model_name = "RazyDave/deberta-v3-base-finetuned-rte"

print(f"Loading: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

print(f"Model: {model.__class__.__name__}")
print(f"Hidden size (H): {model.config.hidden_size}")
print(f"Num layers: {model.config.num_hidden_layers}")

In [ ]:
# Load full RTE training set
dataset = load_dataset("glue", "rte", split="train")
print(f"Loaded {len(dataset)} samples (full training set)")

# Example
print(f"\nExample sentence pair:")
print(f"  S1: {dataset[0]['sentence1']}")
print(f"  S2: {dataset[0]['sentence2']}")

## Define Tokenization

In [ ]:
def tokenize_fn(tokenizer, batch, max_length):
    return tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding="longest",
        truncation=True,
        max_length=max_length,
    )

## Extract Features: Pooling Methods (Baseline)

In [ ]:
# Initialize extractor
extractor = FeaturesExtraction(model, tokenizer)

MAX_LENGTH = 128
BATCH_SIZE = 8

print(f"Model has {model.config.num_hidden_layers} encoder layers")
print("Will extract features from ALL layers")

In [ ]:
print("Extracting with MEAN pooling from ALL layers (baseline)...")
features_dict_mean, labels = extractor.extract_all_layers(
    dataset=dataset,
    tokenize_fn=tokenize_fn,
    config=ExtractionConfig(
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        pooling="mean",
        return_numpy=True,
    ),
)

print(f"\nMean pooling result:")
print(f"  Number of layers: {len(features_dict_mean)}")
print(f"  Layer names: {list(features_dict_mean.keys())[:3]}... (showing first 3)")
# Get first layer to show shape
first_layer_name = list(features_dict_mean.keys())[0]
print(f"  Shape per layer: {features_dict_mean[first_layer_name].shape}")
print(f"  Per sentence per layer: {features_dict_mean[first_layer_name].shape[1]} features (H dimension)")

# Calculate total memory
total_memory = sum(f.nbytes for f in features_dict_mean.values())
print(f"  Total memory (all layers): {total_memory / 1024:.2f} KB")

## Extract Features: FLATTEN (New Experiment)

In [ ]:
print("Extracting with FLATTEN pooling from ALL layers (new)...")
features_dict_flatten, labels = extractor.extract_all_layers(
    dataset=dataset,
    tokenize_fn=tokenize_fn,
    config=ExtractionConfig(
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        pooling="flatten",  # NEW!
        return_numpy=True,
    ),
)

print(f"\nFlatten pooling result:")
print(f"  Number of layers: {len(features_dict_flatten)}")
print(f"  Layer names: {list(features_dict_flatten.keys())[:3]}... (showing first 3)")
# Get first layer to show shape
first_layer_name = list(features_dict_flatten.keys())[0]
print(f"  Shape per layer: {features_dict_flatten[first_layer_name].shape}")
print(f"  Per sentence per layer: {features_dict_flatten[first_layer_name].shape[1]} features (T×H dimension)")
print(f"  Calculation: {MAX_LENGTH} tokens × {model.config.hidden_size} hidden = {MAX_LENGTH * model.config.hidden_size}")

# Calculate total memory
total_memory = sum(f.nbytes for f in features_dict_flatten.values())
print(f"  Total memory (all layers): {total_memory / 1024:.2f} KB")

## Comparison

In [ ]:
print("=" * 70)
print("COMPARISON")
print("=" * 70)

# Get one layer for comparison
sample_layer = list(features_dict_mean.keys())[0]

print(f"\nPer layer comparison (using '{sample_layer}' as example):")
print(f"\nMean Pooling:")
print(f"  Each sentence = {features_dict_mean[sample_layer].shape[1]} features")
print(f"  Number of layers = {len(features_dict_mean)}")
print(f"  Total features per sentence (all layers) = {features_dict_mean[sample_layer].shape[1] * len(features_dict_mean)}")

print(f"\nFlatten (No Pooling):")
print(f"  Each sentence = {features_dict_flatten[sample_layer].shape[1]} features")
print(f"  Number of layers = {len(features_dict_flatten)}")
print(f"  Total features per sentence (all layers) = {features_dict_flatten[sample_layer].shape[1] * len(features_dict_flatten)}")
print(f"  Expansion ratio per layer = {features_dict_flatten[sample_layer].shape[1] / features_dict_mean[sample_layer].shape[1]:.1f}x")

print(f"\nInformation preserved:")
print(f"  Mean pooling: Aggregated (some loss)")
print(f"  Flatten: 100% (no aggregation)")

print(f"\nTotal memory:")
mem_mean = sum(f.nbytes for f in features_dict_mean.values()) / 1024
mem_flatten = sum(f.nbytes for f in features_dict_flatten.values()) / 1024
print(f"  Mean pooling (all layers): {mem_mean:.2f} KB")
print(f"  Flatten (all layers): {mem_flatten:.2f} KB")
print(f"  Memory ratio: {mem_flatten / mem_mean:.1f}x")

## Visualize Sample

In [ ]:
import matplotlib.pyplot as plt

sample_idx = 0
# Use last layer for visualization
last_layer = list(features_dict_mean.keys())[-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Mean pooled
axes[0].bar(range(len(features_dict_mean[last_layer][sample_idx])), 
            features_dict_mean[last_layer][sample_idx], width=1.0)
axes[0].set_title(f"Mean Pooling - {last_layer} (Sample {sample_idx})")
axes[0].set_xlabel(f"Feature Index (0-{len(features_dict_mean[last_layer][sample_idx])-1})")
axes[0].set_ylabel("Value")

# Flattened (show first 768 for comparison)
axes[1].plot(features_dict_flatten[last_layer][sample_idx][:768], linewidth=0.5)
axes[1].set_title(f"Flatten - {last_layer} - First 768 features (Sample {sample_idx})")
axes[1].set_xlabel(f"Feature Index (0-767 of {len(features_dict_flatten[last_layer][sample_idx])})")
axes[1].set_ylabel("Value")

plt.tight_layout()
plt.show()

print(f"\nNote: Flatten has {features_dict_flatten[last_layer].shape[1]} features per sentence per layer!")
print(f"Total layers: {len(features_dict_flatten)}")

## Save Results

In [ ]:
output_dir = REPO_ROOT / "data" / "features"
output_dir.mkdir(parents=True, exist_ok=True)

# Save flatten features (all layers) to separate file
flatten_file = output_dir / "deberta_rte_alllayers_flatten.npz"
np.savez_compressed(
    flatten_file,
    **features_dict_flatten,
    labels=labels,
)
print(f"✓ Saved flatten features (all layers): {flatten_file}")
print(f"  Layers saved: {list(features_dict_flatten.keys())}")

# Save mean features (all layers) to separate file
mean_file = output_dir / "deberta_rte_alllayers_mean.npz"
np.savez_compressed(
    mean_file,
    **features_dict_mean,
    labels=labels,
)
print(f"✓ Saved mean features (all layers): {mean_file}")
print(f"  Layers saved: {list(features_dict_mean.keys())}")

print(f"\nOutput directory: {output_dir}")

## Summary

**Traditional approach (pooling):**
- Sentence → 768 features (H dimension)
- Information is aggregated (mean/max/cls)
- Smaller memory footprint

**New approach (flatten):**
- Sentence → 98,304 features (T×H = 128×768)
- **ALL token information preserved**
- 128× more features per sentence
- No information loss from aggregation

**Use cases for flatten:**
- Meta-feature extraction on full sequence
- Traditional ML models (SVM, Random Forest)
- When you need complete token-level information in a flat format